### **Imports + configuration**

In [1]:
import pandas as pd
from pathlib import Path

DATA_CLEAN = Path("..") / "data_clean"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

### **Chargement des fichiers propres**

In [2]:
df_1930_2014 = pd.read_csv(DATA_CLEAN / "matches_1930_2014_clean.csv")
df_2014 = pd.read_csv(DATA_CLEAN / "matches_2014_clean.csv")
df_2018 = pd.read_csv(DATA_CLEAN / "matches_2018_clean.csv")
df_2022 = pd.read_csv(DATA_CLEAN / "matches_2022_clean.csv")

df_1930_2014.head(), df_2014.head(), df_2018.head(), df_2022.head()

(       home_team away_team  home_result  away_result         result  \
 0         France    Mexico          4.0          1.0         France   
 1  United States   Belgium          3.0          0.0  United States   
 2         Serbia    Brazil          2.0          1.0         Serbia   
 3        Romania      Peru          3.0          1.0        Romania   
 4      Argentina    France          1.0          0.0      Argentina   
 
          date        round        city  stadium       edition  
 0  1930-01-01  group_stage  Montevideo      NaN  1930-URUGUAY  
 1  1930-01-01  group_stage  Montevideo      NaN  1930-URUGUAY  
 2  1930-01-01  group_stage  Montevideo      NaN  1930-URUGUAY  
 3  1930-01-01  group_stage  Montevideo      NaN  1930-URUGUAY  
 4  1930-01-01  group_stage  Montevideo      NaN  1930-URUGUAY  ,
   home_team    away_team  home_result  away_result       result  \
 0    Brazil      Croatia            3            1       Brazil   
 1    Mexico     Cameroon            1 

### **Vérification du schéma**

In [3]:
datasets = {
    "1930_2014": df_1930_2014,
    "2014_finals": df_2014,
    "2018": df_2018,
    "2022": df_2022
}

for name, df in datasets.items():
    print(f"Colonnes {name} :", df.columns.tolist())

# Vérification stricte
all_same_schema = len({tuple(df.columns) for df in datasets.values()}) == 1
all_same_schema

Colonnes 1930_2014 : ['home_team', 'away_team', 'home_result', 'away_result', 'result', 'date', 'round', 'city', 'stadium', 'edition']
Colonnes 2014_finals : ['home_team', 'away_team', 'home_result', 'away_result', 'result', 'date', 'round', 'city', 'stadium', 'edition']
Colonnes 2018 : ['home_team', 'away_team', 'home_result', 'away_result', 'result', 'date', 'round', 'city', 'stadium', 'edition']
Colonnes 2022 : ['home_team', 'away_team', 'home_result', 'away_result', 'result', 'date', 'round', 'city', 'stadium', 'edition']


True

### **Fusion des datasets**

In [4]:
# Supprimer les lignes 2014 du dataset 1930–2014
df_1930_2014 = df_1930_2014[df_1930_2014["edition"] != "2014-BRAZIL"]

df_all = pd.concat(
    [df_1930_2014, df_2014, df_2018, df_2022],
    ignore_index=True
)

df_all.head()

,home_team,away_team,home_result,away_result,result,date,round,city,stadium,edition
0,France,Mexico,4.0,1.0,France,1930-01-01,group_stage,Montevideo,NaN,1930-URUGUAY
1,United States,Belgium,3.0,0.0,United States,1930-01-01,group_stage,Montevideo,NaN,1930-URUGUAY
2,Serbia,Brazil,2.0,1.0,Serbia,1930-01-01,group_stage,Montevideo,NaN,1930-URUGUAY
3,Romania,Peru,3.0,1.0,Romania,1930-01-01,group_stage,Montevideo,NaN,1930-URUGUAY
4,Argentina,France,1.0,0.0,Argentina,1930-01-01,group_stage,Montevideo,NaN,1930-URUGUAY


### **Vérifications de qualité globales**

#### **Inspection s'il y a des NaN**

In [5]:
df_all.isna().sum()

home_team         0
away_team         0
home_result       0
away_result       0
result            0
date              0
round             0
city              0
stadium        6415
edition           0
dtype: int64

#### **Inspection s'il y a des doublons**

In [6]:
df_all.duplicated().sum()

np.int64(1)

#### **Vérification des types**

In [7]:
df_all.dtypes

home_team          str
away_team          str
home_result    float64
away_result    float64
result             str
date               str
round              str
city               str
stadium         object
edition            str
dtype: object

#### **Inspection s'il y a des scores négatifs**

In [8]:
df_all[(df_all["home_result"] < 0) | (df_all["away_result"] < 0)]

,home_team,away_team,home_result,away_result,result,date,round,city,stadium,edition


#### **Vérification que les éditions sont présentes**

In [9]:
df_all["edition"].value_counts()

edition
2010-SOUTH AFRICA    917
2006-GERMANY         911
2002-KOREA/JAPAN     841
1998-FRANCE          708
1994-USA             550
1990-ITALY           367
1986-MEXICO          364
1982-SPAIN           358
1978-ARGENTINA       290
1974-FRG             263
1970-MEXICO          204
1966-ENGLAND         159
1958-SWEDEN          125
1962-CHILE           124
1954-SWITZERLAND      83
2014-BRAZIL           64
2018-RUSSIA           64
2022-QATAR            64
1950-BRAZIL           50
1934-ITALY            43
1938-FRANCE           40
1930-URUGUAY          18
Name: count, dtype: int64

### **Export du dataset fusionné**

In [10]:
output_csv = DATA_CLEAN / "matches_1930_2022_clean.csv"
output_parquet = DATA_CLEAN / "matches_1930_2022_clean.parquet"

df_all.to_csv(output_csv, index=False)
df_all.to_parquet(output_parquet, index=False)